# <center>  **<span style="font-size:80px;">Visualización</span>** </center>

In [ ]:
import plotly.graph_objects as go
import geopandas as gpd
import pandas as pd
import sys
import os

from pathlib import Path

sys.path.append(os.path.abspath(os.path.join("..")))
from src.utils import load_json
from src.config import DatasetKeys, Paths
Paths.init_project()

In [ ]:
mapas_path = Path("./AMAEM/mapas/")
mapas_path.mkdir(parents=True, exist_ok=True)

# Visualización AMAEM JSON

In [ ]:
# 1. Cargamos TODOS los datos en un diccionario: {Path: contenido_json}
# Esto usa los atributos reales de tu clase Paths
all_json_paths = [attr for attr in dir(Paths) if attr.startswith("RAW_JSON_") and attr != "RAW_JSON_DIR"]
data_map = {getattr(Paths, name): load_json(getattr(Paths, name)) for name in all_json_paths}

In [ ]:
# Definimos qué variables de tu clase van a cada grupo
POLIGONOS = [
    Paths.RAW_JSON_BOCAS_HIDRANTES, Paths.RAW_JSON_DEPOSITOS, Paths.RAW_JSON_CENTROS_BOMBEO,
    Paths.RAW_JSON_ENTIDADES_POBLACION, Paths.RAW_JSON_SECTORES_CONSUMO, Paths.RAW_JSON_ZONAS_VERDES
]

LINEAS = [
    Paths.RAW_JSON_REDES_ARTERIALES, Paths.RAW_JSON_REDES_PRIMARIAS, Paths.RAW_JSON_GRANDES_COLECTORES,
    Paths.RAW_JSON_TUBERIAS_REGENERADA, Paths.RAW_JSON_TUBERIAS_ALCANTARILLADO, 
    Paths.RAW_JSON_FUENTES, Paths.RAW_JSON_IMBORNALES, Paths.RAW_JSON_IMBORNALES_GRAN, Paths.RAW_JSON_PLUVIOMETROS
]

In [ ]:
# ==========================================
# 1. CONFIGURACIÓN DE COLORES (Suaves / Pastel)
# ==========================================
# rgba(R, G, B, Transparencia)
ESTILOS_POLIGONOS = {
    "Zonas Verdes": dict(fill='rgba(130, 160, 130, 0.4)', line='rgba(100, 130, 100, 0.6)'), # Verde grisáceo
    "Sectores De Consumo": dict(fill='rgba(120, 150, 180, 0.3)', line='rgba(100, 130, 160, 0.5)'), # Azul grisáceo
    "Entidades De Poblacion": dict(fill='rgba(180, 160, 130, 0.3)', line='rgba(160, 140, 110, 0.5)'), # Marrón/Crema grisáceo
    "Default": dict(fill='rgba(200, 200, 200, 0.2)', line='rgba(150, 150, 150, 0.4)') # Gris sutil para el resto
}

ESTILOS_LINEAS = {
    "Redes Arteriales": 'rgba(200, 100, 100, 0.8)', # Rojo desaturado
    "Redes Primarias": 'rgba(200, 150, 100, 0.8)', # Naranja desaturado
    "Tuberias De Alcantarillado Y Pluviales": 'rgba(130, 130, 130, 0.8)', # Gris oscuro
    "Default": 'rgba(150, 150, 180, 0.8)' # Gris azulado por defecto
}

# 2. Creamos el "Motor"
fig = go.Figure()

# Nivel de detalle (2 = usa la mitad de los puntos. 1 = usa todos. Sube a 3 si va muy lento)
STEP = 2 

# --- DIBUJAR POLÍGONOS ---
for path in POLIGONOS:
    geojson = data_map[path]
    label = path.stem.replace('-', ' ').title()
    
    # Buscamos su color, si no existe usamos el "Default"
    estilo = ESTILOS_POLIGONOS.get(label, ESTILOS_POLIGONOS["Default"])
    
    x_all, y_all = [], []
    
    for feature in geojson['features']:
        for ring in feature['geometry'].get('rings', []):
            x, y = zip(*ring)
            
            # [::STEP] filtra los puntos para que el navegador no sufra
            x_all.extend(x[::STEP])
            x_all.append(None)
            y_all.extend(y[::STEP])
            y_all.append(None)
            
    if x_all:
        fig.add_trace(go.Scatter( # Scatter normal soporta mejor el relleno 'toself'
            x=x_all, y=y_all,
            fill='toself',
            fillcolor=estilo['fill'],
            mode='lines',
            line=dict(width=1, color=estilo['line']),
            name=label,
            hoverinfo='name'
        ))

# --- DIBUJAR LÍNEAS (ACELERADO POR WEBGL) ---
for path in LINEAS:
    geojson = data_map[path]
    label = path.stem.replace('-', ' ').title()
    
    color_linea = ESTILOS_LINEAS.get(label, ESTILOS_LINEAS["Default"])
    
    x_all, y_all = [], []
    
    for feature in geojson['features']:
        for p_path in feature['geometry'].get('paths', []):
            x, y = zip(*p_path)
            
            x_all.extend(x[::STEP])
            x_all.append(None)
            y_all.extend(y[::STEP])
            y_all.append(None)
            
    if x_all:
        # ATENCIÓN: Usamos Scattergl en lugar de Scatter. ¡Esto usa tu Tarjeta Gráfica!
        fig.add_trace(go.Scattergl(
            x=x_all, y=y_all,
            mode='lines',
            line=dict(width=1.5, color=color_linea),
            name=label,
            hoverinfo='name'
        ))

# 3. Configuración del "Motor"
fig.update_layout(
    height=900,
    width=1400,
    title='Infraestructura Hidráulica Interactiva (Optimizada con WebGL)',
    title_font_size=20,
    yaxis=dict(scaleanchor="x", scaleratio=1),
    plot_bgcolor='#F8F9FA', # Un fondo un pelín gris muy claro para que destaquen las líneas claras
    legend=dict(
        title="Capas (Clic para filtrar)",
        itemsizing='constant'
    ),
    dragmode='pan',
    hovermode='closest'
)

fig.write_html(mapas_path / "alicante_interactivo.html")
fig.show()

# Visualización Resultados

In [ ]:
def generar_mapa_interactivo(
    ruta_csv,
    ruta_geojson,
    variable_mapa,
    columnas_tooltip,
    agregaciones,
    nombre_archivo_salida,
    cmap="YlOrRd",
    convertir_variable_a_entero=False
):
    """
    Genera un mapa interactivo (Folium) cruzando datos tabulares con geometrías reales.
    """
    # 1. CARGA Y AGREGACIÓN DE DATOS
    df_raw = pd.read_csv(ruta_csv)

    # Filtramos solo las columnas que realmente existan en el CSV
    dict_agg = {k: v for k, v in agregaciones.items() if k in df_raw.columns}

    # Agrupamos por barrio para tener una única fila por polígono
    df_datos = df_raw.groupby(DatasetKeys.BARRIO).agg(dict_agg).reset_index()

    # Limpiamos el nombre del barrio (ej: "10-FLORIDA BAJA" -> "FLORIDA BAJA")
    df_datos['barrio_limpio'] = df_datos[DatasetKeys.BARRIO].str.split('-', n=1).str[-1].str.strip()

    # 2. CARGAMOS GEOMETRÍAS REALES
    # Usamos directamente geopandas para leer el archivo JSON de Entidades
    gdf_areas = gpd.read_file(ruta_geojson)

    # 3. CRUCE DE DATOS Y GEOMETRÍA
    gdf_final = gdf_areas.merge(
        df_datos, 
        left_on='DENOMINACI', 
        right_on='barrio_limpio', 
        how='inner'
    )

    # ConvertiMOS de UTM (25830) a GPS (4326) para que Folium lo posicione bien
    gdf_final = gdf_final.to_crs(epsg=4326)

    # Calculamos el percentil 95 para tratar los supuestos 'Outliers'
    #vmax = gdf_final[variable_mapa].quantile(0.95)
    vmax = gdf_final[variable_mapa].max()

    # Transformación opcional: Convertir la variable a entero si el usuario lo pide
    if convertir_variable_a_entero:
        gdf_final[variable_mapa] = gdf_final[variable_mapa].astype(int)

    # 4. VISUALIZACIÓN (Folium / Leaflet)
    mapa_hackathon = gdf_final.explore(
        column=variable_mapa,
        cmap=cmap,                # Paleta configurable
        vmin=0,                   
        vmax=vmax,                
        tooltip=columnas_tooltip, # Tooltips configurables
        popup=True,               
        tiles="CartoDB dark_matter", 
        style_kwds={              
            "fillOpacity": 0.6,   
            "weight": 0.8,        
            "color": "#444444"    
        },
        highlight_kwds={          
            "fillOpacity": 0.9, 
            "weight": 2, 
            "color": "#D6D8D8"
        },
        name="Infraestructura y Consumo"
    )

    # Guardamos el resultado
    ruta_salida = mapas_path / nombre_archivo_salida
    mapa_hackathon.save(ruta_salida)

    print(f"Mapa '{nombre_archivo_salida}' generado con éxito en: {ruta_salida}")
    
    return mapa_hackathon

## Ejemplo 1: Mapa del Consumo Total

In [ ]:
agregaciones_base = {
    DatasetKeys.CONSUMO: 'sum',
    DatasetKeys.NUM_CONTRATOS: 'sum',
    DatasetKeys.CONSUMO_RATIO: 'mean',
    DatasetKeys.PCT_VT_BARRIO: 'mean',
    DatasetKeys.NUM_VT_BARRIO: 'sum'
}

mapa_consumo = generar_mapa_interactivo(
    ruta_csv=Paths.PROC_CSV_AMAEM_NOT_SCALED,
    ruta_geojson=Paths.RAW_JSON_ENTIDADES_POBLACION,
    variable_mapa=DatasetKeys.CONSUMO,
    columnas_tooltip=["DENOMINACI", DatasetKeys.CONSUMO, DatasetKeys.NUM_CONTRATOS],
    agregaciones=agregaciones_base,
    nombre_archivo_salida="mapa_consumo_total.html",
    cmap="YlGnBu", 
    convertir_variable_a_entero=True # El consumo tiene sentido como entero
)

## Ejemplo 2: Mapa de Presión Turística (Viviendas Turísticas)

In [ ]:
mapa_turismo = generar_mapa_interactivo(
    ruta_csv=Paths.PROC_CSV_AMAEM_NOT_SCALED,
    ruta_geojson=Paths.RAW_JSON_ENTIDADES_POBLACION,
    variable_mapa=DatasetKeys.PCT_VT_BARRIO,
    columnas_tooltip=["DENOMINACI", DatasetKeys.PCT_VT_BARRIO, DatasetKeys.NUM_VT_BARRIO],
    agregaciones=agregaciones_base,
    nombre_archivo_salida="mapa_presion_turistica.html",
    cmap="plasma", # Resalta mucho las zonas turísticas
    convertir_variable_a_entero=False # Mantenemos los decimales para el porcentaje
)